<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Principal Component Analysis (PCA)

Throughout, let

$$
X \in \mathbb{R}^{N\times D}
$$

be the data matrix. Rows are observations and columns are variables.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import eigh, norm
from IPython.display import display

np.set_printoptions(precision=4, suppress=True)


def unit(v):
    """Return a unit-length version of a vector."""
    v = np.asarray(v, dtype=float)
    return v / norm(v)


In [ ]:
def orient_eigenvectors(V):
    """Choose a deterministic sign convention for eigenvectors.

    Eigenvectors are only defined up to sign. This function flips each
    eigenvector so that its largest absolute entry is positive.
    """
    V = V.copy()
    for j in range(V.shape[1]):
        idx = np.argmax(np.abs(V[:, j]))
        if V[idx, j] < 0:
            V[:, j] *= -1
    return V


def pca_from_covariance(X, q=None, center=True):
    """Compute PCA by forming the covariance matrix and finding its eigenpairs.

    Parameters
    ----------
    X : array-like, shape (N, D)
        Data matrix with observations in rows and variables in columns.
    q : int or None
        Number of principal components to return. If None, return all.
    center : bool
        Whether to subtract column means before forming the covariance matrix.
    """
    X = np.asarray(X, dtype=float)
    N, D = X.shape

    if center:
        mean = X.mean(axis=0)
        X0 = X - mean
    else:
        mean = np.zeros(D)
        X0 = X.copy()

    S_X = (X0.T @ X0) / (N - 1)

    # eigh is for symmetric matrices. It returns eigenvalues in ascending order.
    eigenvalues, eigenvectors = eigh(S_X)
    order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # Tiny negative values can occur from floating-point error.
    eigenvalues = np.maximum(eigenvalues, 0)
    eigenvectors = orient_eigenvectors(eigenvectors)

    if q is None:
        q = D

    W = eigenvectors[:, :q]
    scores = X0 @ W

    return {
        "mean": mean,
        "centered_data": X0,
        "covariance": S_X,
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "W": W,
        "scores": scores,
    }

PCA is a dimensionality reduction technique.

The **first goal** is to replace variables

$$
X_1,\ldots,X_D
$$

with new ones

$$
Z_1,\ldots,Z_q,
$$

where

$$
q \ll D.
$$

We will call these $Z_i$ the **principal component scores**.

The **second goal** is to avoid losing too much "information".

The central dogma of PCA is that:

$$
\text{variance} \approx \text{information}.
$$

So PCA tries to find new variables with large variance.

When is this reasonable? Example:

In [ ]:
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(657677)
n = 600

Z = rng.normal(size=(n, 3), scale=1)
Z[:, 0] *= 3.0
Z[:, 1] *= 1.2
Z[:, 2] *= 0.5

theta = np.pi / 5
phi = np.pi / 7

Rz = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0,              0,             1]
])

Ry = np.array([
    [ np.cos(phi), 0, np.sin(phi)],
    [0,            1, 0],
    [-np.sin(phi), 0, np.cos(phi)]
])

R = Rz @ Ry
X = Z @ R.T

# Mean-center
xbar = X.mean(axis=0)
X_centered = X - xbar

pca_3d = pca_from_covariance(X, q=2, center=True)
V = pca_3d["eigenvectors"]
eigenvalues = pca_3d["eigenvalues"]
scores = pca_3d["scores"]

X_projected_3d = scores[:, :2] @ V[:, :2].T + xbar

In [ ]:
# Make a grid in PC1-PC2 coordinates
grid_size = 25
pc1_grid = np.linspace(scores[:, 0].min(), scores[:, 0].max(), grid_size)
pc2_grid = np.linspace(scores[:, 1].min(), scores[:, 1].max(), grid_size)

A, B = np.meshgrid(pc1_grid, pc2_grid)

plane_scores = np.column_stack([
    A.ravel(),
    B.ravel()
])

plane_3d = plane_scores @ V[:, :2].T + xbar

PX = plane_3d[:, 0].reshape(grid_size, grid_size)
PY = plane_3d[:, 1].reshape(grid_size, grid_size)
PZ = plane_3d[:, 2].reshape(grid_size, grid_size)

lines = []
scale = 3.5

for j in range(2):
    direction = V[:, j]
    endpoint_1 = xbar - scale * np.sqrt(eigenvalues[j]) * direction
    endpoint_2 = xbar + scale * np.sqrt(eigenvalues[j]) * direction

    lines.append(
        go.Scatter3d(
            x=[endpoint_1[0], endpoint_2[0]],
            y=[endpoint_1[1], endpoint_2[1]],
            z=[endpoint_1[2], endpoint_2[2]],
            mode="lines",
            line=dict(width=8),
            name=f"PC{j + 1}"
        )
    )

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X[:, 0],
        y=X[:, 1],
        z=X[:, 2],
        mode="markers",
        marker=dict(size=3, opacity=0.35),
        name="Original 3D data"
    )
)

fig.add_trace(
    go.Scatter3d(
        x=X_projected_3d[:, 0],
        y=X_projected_3d[:, 1],
        z=X_projected_3d[:, 2],
        mode="markers",
        marker=dict(size=3, opacity=0.35),
        name="Projected onto PCA plane"
    )
)

fig.add_trace(
    go.Surface(
        x=PX,
        y=PY,
        z=PZ,
        opacity=0.25,
        showscale=False,
        name="PCA plane"
    )
)

for line in lines:
    fig.add_trace(line)

fig.update_layout(
    title="Interactive 3D pancake cloud and projection onto the first two PCs",
    scene=dict(
        xaxis_title="$X_1$",
        yaxis_title="$X_2$",
        zaxis_title="$X_3$",
        aspectmode="data"
    ),
    width=900,
    height=700,
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig.show()

Here is it projected:

In [ ]:
import matplotlib.pyplot as plt

scores_2d = scores[:, :2]

plt.figure(figsize=(7, 6))

plt.scatter(
    scores_2d[:, 0],
    scores_2d[:, 1],
    s=14,
    alpha=0.55
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("$z_1 = X w_1$")
plt.ylabel("$z_2 = X w_2$")
plt.title("Projected data in the first two principal component directions")

plt.gca().set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

Here, we are visualizing $X$ row-wise so that the $n^{\text{th}}$ row is

$$
x_n \in \mathbb{R}^D.
$$

PCA tries to keep this simple, it looks at **linear combinations** of the original variables $X_1,\ldots,X_D$ to create $Z_1,\ldots,Z_q$. Equivalently, the goal is to find a lower-dimensional (linear) **subspace** of $\mathbb{R}^D$ so that when we project the data onto this subspace we don't lose too much information. 

## Projection matrices

Let's do a refresher on projection matrices. Let $W$ be the $D\times q$ matrix of basis elements for the lower-dimensional subspace. 

The projection matrix onto $\operatorname{Col}(W)$ is

$$
P_W = W(W^T W)^{-1}W^T \in \mathbb{R}^{D\times D}.
$$

If $x\in\mathbb{R}^{1 \times D}$ is a row-vector, then

$$
xP_W \in\mathbb{R}^D
$$

is the projected point, still written in the original $\mathbb{R}^D$ coordinates.

For this lecture, WLOG, we can assume that the columns of $W$ are orthonormal so that we have an orthonormal basis (Why not?). In this case, 

$$
W^TW=I_q,
$$

(why?) so

$$
P_W=WW^T.
$$

Then

$$
xP_W=xWW^T.
$$

If we want to project all of the data points (rows) in $X\in\mathbb{R}^{N\times D}$ onto $\operatorname{Col}(W)$ then we can write it as: 

$$
XP_W = XWW^T \in\mathbb{R}^{N \times D}
$$

This would be our original data points but now projected down onto the $q$-dimensional subspace. 

There are two related objects:

$$
Z = XW \in \mathbb{R}^{N\times q}
$$

is the data expressed in the new $q$-dimensional coordinates, while

$$
XP_W = XWW^T \in \mathbb{R}^{N\times D}
$$

is the projected data written back in the original $D$ coordinates.

## Principal components as linear combinations

Since PCA wants to find positions of the data points in the lower-dimensional space (i.e using only $q$ variables), then we are mostly interested in 
$$
Z=XW
$$
i.e., the data matrix embedded in the lower-dimensional coordinates.

Notice that if $Z_i$ is the $i^{\text{th}}$ column (variable) of $Z$, $X_j$ is the $j^{\text{th}}$ column (variable) of $X$, and $w_i$ is the $i^{\text{th}}$ column of $W$, then

$$
Z_i = Xw_i
$$

and therefore

$$
Z_i = X_1w_{i1}+X_2w_{i2}+\cdots+X_Pw_{iD}.
$$

So, *equivalently*, each principal component is a linear combination of the original columns of $X$.

## PCA objective

With all this in mind, one way to frame PCA is: it finds linear combinations of the columns (variables) of $X$ such that

1. the variances of the resulting $Z_i$ variables are as large as possible (recall: want to retain information $\approx$ variance)
2. the $Z_i$ variables are uncorrelated (so there is no redundancy)
3. the $w_i$ vectors have unit length (a practical constraint, otherwise variance could grow without bound)

For example, to find the first PC we could sweep over all possible vectors:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

rng = np.random.default_rng(657677)
n = 300

# Start with independent coordinates with unequal variances
Z = rng.normal(size=(n, 2))
Z[:, 0] *= 3.0
Z[:, 1] *= 0.8

# Rotate the cloud so the main direction is not axis-aligned
theta = np.pi / 5

R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

X = Z @ R.T

# Mean-center the data before applying PCA
X = X - X.mean(axis=0)

# Compute projected variance over all directions
angles = np.linspace(0, np.pi, 361)

variances = []

for a in angles:
    w_a = np.array([np.cos(a), np.sin(a)])
    z_a = X @ w_a
    variances.append(np.var(z_a, ddof=1))

variances = np.array(variances)

best_angle = angles[np.argmax(variances)]
best_angle_deg = np.rad2deg(best_angle)
best_w = np.array([np.cos(best_angle), np.sin(best_angle)])

print("Best direction angle, degrees:", best_angle_deg.round(2))
print("Best unit vector:", best_w.round(4))
print("Max projected variance:", variances.max().round(4))


# Interactive display
angle_slider = widgets.FloatSlider(
    value=best_angle_deg,
    min=0,
    max=180,
    step=1,
    description="angle",
    continuous_update=True,
    readout_format=".0f"
)

out = widgets.Output()

xlim = 1.15 * np.max(np.abs(X[:, 0]))
ylim = 1.15 * np.max(np.abs(X[:, 1]))
line_t = np.linspace(-4, 4, 100)

def plot_for_angle(angle_deg):
    a = np.deg2rad(angle_deg)
    w = np.array([np.cos(a), np.sin(a)])
    z = X @ w
    current_variance = np.var(z, ddof=1)

    line = line_t[:, None] * w[None, :]

    with out:
        clear_output(wait=True)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Left plot: projected variance as a function of angle
        axes[0].plot(np.rad2deg(angles), variances)
        axes[0].scatter([angle_deg], [current_variance], s=80, zorder=3)

        axes[0].axvline(best_angle_deg, linestyle="--", linewidth=1)
        axes[0].set_xlabel("direction angle in degrees")
        axes[0].set_ylabel(r"sample variance of $Xw$")
        axes[0].set_title("Projected variance by direction")

        axes[0].text(
            0.03,
            0.95,
            f"current variance = {current_variance:.3f}\nbest angle = {best_angle_deg:.1f}°",
            transform=axes[0].transAxes,
            va="top"
        )

        # Right plot: data cloud and selected direction
        axes[1].scatter(X[:, 0], X[:, 1], s=18, alpha=0.45)
        axes[1].plot(line[:, 0], line[:, 1], linewidth=3, label="selected direction")

        # Optional: show projections of points onto the selected direction
        projected = np.outer(z, w)
        axes[1].scatter(projected[:, 0], projected[:, 1], s=10, alpha=0.25)

        axes[1].axhline(0, linewidth=1)
        axes[1].axvline(0, linewidth=1)

        axes[1].set_xlim(-xlim, xlim)
        axes[1].set_ylim(-ylim, ylim)
        axes[1].set_xlabel("$x_1$")
        axes[1].set_ylabel("$x_2$")
        axes[1].set_title("Data cloud and selected direction")
        axes[1].set_aspect("equal", adjustable="box")
        axes[1].legend()

        plt.tight_layout()
        plt.show()

def update(change):
    plot_for_angle(change["new"])

angle_slider.observe(update, names="value")

display(angle_slider, out)
plot_for_angle(angle_slider.value)

**Ok, so how do we do this generally?**

## Aside: variance and covariance notation

Let $x\in\mathbb{R}^N$ be a variable observed on $N$ units. (**Note** this is a *column* of $X$ here)

Assume

$$
\bar{x}=\frac{1}{N}\sum_{n=1}^N x_n=0.
$$
(if not, mean-center the variable). Then

$$
\widehat{\operatorname{Var}}(x)
=\frac{1}{N-1}\sum_n (x_n-\bar{x})^2
=\frac{1}{N-1}\sum_n x_n^2
\propto x^Tx.
$$

Similarly, if $y\in\mathbb{R}^N$ is another centered variable, then

$$
\widehat{\operatorname{Cov}}(x,y)
=\frac{1}{N-1}x^Ty \propto x^Ty
$$

So for centered variables:

$$
\text{uncorrelated} \quad \Longleftrightarrow \quad \text{orthogonal}.
$$

If $X$ is an $N\times D$ data matrix then the sample covariance matrix the is a $D\times D$ matrix $S$ where

$$
S_{ij}=\widehat{\operatorname{Cov}}(X_i,X_j),
\qquad
S_{ii}=\widehat{\operatorname{Var}}(X_i).
$$
It encodes all of the variances of the variables and all of the covariances amongt the variables. Its rather like a generalization of the variance to a collection of variables. 

If $X$ has mean-centered columns, then

$$
S = \widehat{\operatorname{Cov}}(X)
=\frac{1}{N-1}X^TX \propto X^TX
$$

Note: $S$ is **symmetric** so that $S=S^T$. 

## The PCA Problem

Let $X \in \mathbb{R}^{N \times D}$ be centered, so each column has sample mean zero. Define the sample covariance matrix

$$
S_X = \frac{1}{N-1}X^TX.
$$

For a unit direction $w \in \mathbb{R}^D$, the corresponding score variable is

$$
z = Xw.
$$

Its sample variance is

$$
\operatorname{Var}(Xw)
=\frac{1}{N-1}(Xw)^T(Xw)
=w^TS_Xw.
$$

Thus PCA can be understood as finding directions $w_1,w_2,\dots$ so that the score variables $Xw_1,Xw_2,\dots$ have large variance and are uncorrelated.

More explicitly, the first principal component is defined by the optimization problem

$$
w_1=\arg\max_{\|w\|_2=1}
\operatorname{Var}(Xw).
$$

Using the formula above, this is equivalent to

$$
w_1=\arg\max_{\|w\|_2=1}
w^T S_X w.
$$

For later components, we want the new score to have large variance while being uncorrelated with the previous scores. So, for $k \geq 2$,

$$
w_k=\arg\max_{\|w\|_2=1,\; \operatorname{Cov}(Xw, Xw_j)=0 \text{ for } j<k}
\operatorname{Var}(Xw).
$$

Equivalently,

$$
w_k=\arg\max_{\|w\|_2=1,\; \operatorname{Cov}(Xw, Xw_j)=0 \text{ for } j<k}
w^T S_X w.
$$

So the objective is sequential: each new component maximizes variance subject to unit length and zero covariance with all previously chosen score variables. Since $X$ is centered,

$$
\operatorname{Cov}(Xw, Xw_j)=w^T S_X w_j.
$$

Therefore, the optimization can also be written as

$$
w_k=\arg\max_{\|w\|_2=1,\; w^T S_X w_j=0 \text{ for } j<k}
w^T S_X w.
$$

After the optimization, the **principal component scores** are then

$$
z_k = Xw_k.
$$

### Simplified Rayleigh quotient theorem

Let $A \in \mathbb{R}^{D \times D}$ be symmetric (like a covariance matrix). One can show that this means that it has a basis of real orthonormal eigenvectors and associated real eigen-values. Let the eigenvectors be:

$$
v_1, v_2, \ldots, v_D
$$

with corresponding eigenvalues

$$
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_D.
$$

Then

$$
\max_{\|w\|_2=1} w^TAw = \lambda_1.
$$

The maximum is achieved by

$$
w = v_1,
$$

where $v_1$ is the eigenvector corresponding to $\lambda_1$.

More generally,

$$
\max_{\|w\|_2=1,\; w \perp v_1,\dots,v_{k-1}} w^TAw=\lambda_k,
$$

and the maximum is achieved by

$$
w = v_k.
$$

**Proof.** To see why, write $w$ as a linear combination of the eigenvectors:

$$
w = c_1v_1 + c_2v_2 + \cdots + c_Dv_D.
$$

If $\|w\|_2=1$, then

$$
c_1^2 + c_2^2 + \cdots + c_D^2 = 1.
$$

Now

$$
w^TAw=\lambda_1c_1^2 + \lambda_2c_2^2 + \cdots + \lambda_Dc_D^2.
$$

This is a weighted average of the eigenvalues, with weights $c_i^2$. Since the largest eigenvalue is $\lambda_1$,

$$
w^TAw \leq \lambda_1.
$$

The upper bound is achieved by putting all the weight on the first eigenvector, meaning $w=v_1$.

Similarly, if we require

$$
w \perp v_1,\dots,v_{k-1},
$$

then

$$
c_1 = \cdots = c_{k-1} = 0.
$$

Therefore,

$$
w^TAw=\lambda_kc_k^2 + \lambda_{k+1}c_{k+1}^2 + \cdots + \lambda_Dc_D^2
\leq \lambda_k.
$$

The upper bound is achieved by taking

$$
w = v_k.
$$

### Applying the theorem to PCA

The first principal component solves

$$
w_1=\arg\max_{\|w\|_2=1}
w^TS_Xw.
$$

The covariance matrix $S_X$ is symmetric and has nonnegative eigenvalues. Let

$$
v_1, v_2, \ldots, v_D
$$

be orthonormal eigenvectors of $S_X$, with corresponding eigenvalues

$$
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_D \geq 0.
$$

By the Rayleigh quotient theorem,

$$
w_1 = v_1.
$$

So the first principal direction is the eigenvector of $S_X$ with the largest eigenvalue.

The second principal component should maximize variance while being uncorrelated with the first score. The covariance between $Xw$ and $Xw_1$ is

$$
\operatorname{Cov}(Xw, Xw_1)=w^TS_Xw_1.
$$

Since $w_1=v_1$ and $S_Xv_1=\lambda_1v_1$,

$$
w^TS_Xw_1=w^TS_Xv_1=\lambda_1 w^Tv_1.
$$

Therefore, if $\lambda_1>0$, the condition

$$
\operatorname{Cov}(Xw, Xw_1)=0
$$

is equivalent to

$$
w \perp v_1.
$$

So the second direction solves

$$
w_2=\arg\max_{\|w\|_2=1,\; w \perp v_1}
w^TS_Xw.
$$

By the restricted Rayleigh quotient theorem,

$$
w_2 = v_2.
$$

Continuing in the same way, the $k$th principal direction solves

$$
w_k=\arg\max_{\|w\|_2=1,\; w \perp v_1,\dots,v_{k-1}}
w^TS_Xw.
$$

Therefore,

$$
w_k = v_k.
$$

Thus the first $q$ PCA directions are

$$
W_q =\begin{bmatrix}
v_1 & v_2 & \cdots & v_q
\end{bmatrix}.
$$

## Checking Score Properties

The PCA scores are

$$
Z_q = XW_q.
$$

Since $X$ is centered, $Z_q$ is also centered because each column of $Z_q$ is a linear combination of centered variables.

The covariance matrix of the scores is

$$
S_{Z_q}
=\frac{1}{N-1}Z_q^TZ_q
=W_q^TS_XW_q.
$$

Since $W_q$ contains the first $q$ eigenvectors of $S_X$,

$$
S_{Z_q}
=\operatorname{diag}(\lambda_1,\dots,\lambda_q).
$$

(why?)

Thus the PCA scores are uncorrelated, and their variances are

$$
\lambda_1,\dots,\lambda_q.
$$

So the eigenvalue derivation gives exactly what PCA wants: the scores are uncorrelated, their variances are as large as possible in order, and the directions are unit vectors.

**Caveats**: If two *eigenvalues are equal*, then the corresponding eigenvectors are not uniquely determined inside that tied eigenspace. Also, *signs* are arbitrary: replacing $v_i$ by $-v_i$ only flips the sign of the corresponding score.

## PCA from the covariance matrix

The practical calculation follows directly from the derivation above.

Start with centered data $X$ and form the covariance matrix

$$
S_X = \frac{1}{N-1}X^TX.
$$

Find the eigenvalues and eigenvectors of $S_X$:

$$
S_X v_i = \lambda_i v_i,
\qquad
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_D.
$$

The principal directions are the eigenvectors

$$
w_i = v_i.
$$

The variance of the $i$th principal component score is

$$
\operatorname{Var}(z_i)=\lambda_i.
$$

The score matrix for the first $q$ components is

$$
Z_q = XW_q,
$$

where

$$
W_q =\begin{bmatrix}
v_1 & v_2 & \cdots & v_q
\end{bmatrix}.
$$

### Guideline: PCA by covariance eigenvalues and eigenvectors

So the steps for calculating PCA are:

0. Mean-center the columns of $X$.

1. Form the covariance matrix:

$$
S_X = \frac{1}{N-1}X^TX.
$$

2. Find the eigenvalues and eigenvectors of $S_X$.

3. Sort the eigenvalues from largest to smallest, and sort the eigenvectors in the same order.

4. Set

$$
W_q=\begin{bmatrix}
v_1 & v_2 & \cdots & v_q
\end{bmatrix}.
$$

5. Compute the scores:

$$
Z_q=XW_q.
$$

The columns of $W_q$ are the **loadings** or **principal directions**.

The columns of $Z_q$ are the **principal components** or **scores**.

This is the method we will use below. In code, the only subtle detail is that many numerical routines return eigenvalues in ascending order, so we sort them in decreasing order before selecting the first $q$ components.

In [ ]:
res = pca_from_covariance(X, q=2, center=True)

S_X = res["covariance"]
V = res["eigenvectors"]
eigenvalues = res["eigenvalues"]
W = res["W"]
Z = res["scores"]

print("Covariance matrix S_X:")
print(S_X)

print("\nEigenvectors V, stored in columns:")
print(V)

print("\nEigenvalues:")
print(eigenvalues)

print("\nCovariance of the first two PC scores:")
print(np.cov(Z, rowvar=False, ddof=1))

In [ ]:
# Plot the PCA directions on top of the synthetic data.

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.55)

for j in range(2):
    direction = V[:, j]
    length = 2.0 
    end = length * direction
    plt.arrow(0, 0, end[0], end[1], width=0.03, length_includes_head=True)
    plt.text(end[0] * 1.05, end[1] * 1.05, f"PC{j+1}")

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel(r"$X_1$")
plt.ylabel(r"$X_2$")
plt.title("PCA directions are the high-variance orthogonal axes")
plt.show()

## Total variance and percent captured

One useful metric is the **total variance captured** by the first $q$ PC scores:

$$
\sum_{i=1}^q \widehat{\operatorname{Var}}(z_i)
=\sum_{i=1}^q \lambda_i.
$$

The **proportion of variance captured** by the first $q$ PCs is

$$
\frac{\sum_{i=1}^q\lambda_i}{\sum_{i=1}^D\lambda_i}.
$$

The denominator is the total variance across the original centered variables. The numerator is the variance retained after projecting onto the first $q$ principal component directions.

In [ ]:
explained_variances = eigenvalues

In [ ]:
explained_table = pd.DataFrame({
    "PC": np.arange(1, len(explained_variances) + 1),
    "variance": explained_variances,
    "proportion": explained_variances / np.sum(explained_variances),
    "cumulative_proportion": np.cumsum(explained_variances / np.sum(explained_variances)),
})
display(explained_table)

## Mean-centering matters

If we do not center the columns, the first principal component can mostly describe the location of the data cloud relative to the origin. In that case, $z_1$ may be close to a mean direction rather than the main direction of variation around the mean.

In [ ]:
rng = np.random.default_rng(657677)
n = 500

# A stretched 2D cloud
Z = rng.normal(size=(n, 2))
Z[:, 0] *= 3.0
Z[:, 1] *= 0.5

# Rotate the cloud
theta = np.pi / 5
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

X_centered_true = Z @ R.T

# Add a large mean shift
mu = np.array([-8.0, 6.0])
X_raw = X_centered_true + mu

In [ ]:
# Uncentered: use X_raw^T X_raw instead of the covariance matrix of centered data.
res_uncentered = pca_from_covariance(X_raw, q=1, center=False)
v_uncentered = res_uncentered["W"][:, 0]

# Centered: subtract the column means before forming the covariance matrix.
res_centered = pca_from_covariance(X_raw, q=1, center=True)
v_centered = res_centered["W"][:, 0]

In [ ]:
v_uncentered

In [ ]:
v_centered

In [ ]:
mean_direction = unit(X_raw.mean(axis=0))

print("Absolute dot product: uncentered PC1 with mean direction")
print(abs(v_uncentered @ mean_direction).round(4))

print("\nAbsolute dot product: centered PC1 with mean direction")
print(abs(v_centered @ mean_direction).round(4))

print("\nUncentered PC1:")
print(v_uncentered.round(4))

print("\nCentered PC1:")
print(v_centered.round(4))

print("\nMean direction:")
print(mean_direction.round(4))

## Real data

For a real data example, use the wine data from `sklearn.datasets`.

The variables are measured in different units, so we use PCA on standardized variables. This means PCA is applied to the **correlation structure** rather than the raw **covariance structure**.

After standardization, we form the covariance matrix and compute its eigenvalues and eigenvectors.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine_raw = wine.data.to_numpy()
feature_names = wine.feature_names
target = wine.target.to_numpy()
target_names = wine.target_names

wine_df = wine.frame.copy()
display(wine_df.head())
print("Data shape:", X_wine_raw.shape)
print("Number of classes:", len(target_names))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize each variable before PCA.
scaler = StandardScaler()
X_wine = scaler.fit_transform(X_wine_raw)

In [ ]:
wine_res = pca_from_covariance(X_wine, q=X_wine.shape[1], center=True)

X_wine_scores = wine_res["scores"]
wine_components = wine_res["eigenvectors"]
wine_eigenvalues = wine_res["eigenvalues"]
wine_explained_ratio = wine_eigenvalues / wine_eigenvalues.sum()

In [ ]:
X_wine_scores.shape

In [ ]:
X_wine_scores[:10,:3]

In [ ]:
# The columns of wine_components are the loading vectors.
wine_components.shape

In [ ]:
wine_explained = pd.DataFrame({
    "PC": np.arange(1, len(wine_eigenvalues) + 1),
    "variance": wine_eigenvalues,
    "proportion": wine_explained_ratio,
    "cumulative_proportion": np.cumsum(wine_explained_ratio),
})

display(wine_explained.head(10))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wine_explained["PC"], wine_explained["proportion"], marker="o")
plt.xlabel("principal component")
plt.ylabel("proportion of variance explained")
plt.title("Wine data scree plot")
plt.xticks(wine_explained["PC"])
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wine_explained["PC"], wine_explained["cumulative_proportion"], marker="o")
plt.xlabel("number of PCs retained")
plt.ylabel("cumulative proportion of variance explained")
plt.title("Cumulative variance explained in the wine data")
plt.xticks(wine_explained["PC"])
plt.ylim(0, 1.05)
plt.show()

In [ ]:
Z_wine = X_wine_scores

plt.figure(figsize=(7, 5))

for k, name in enumerate(target_names):
    mask = target == k
    plt.scatter(
        Z_wine[mask, 0],
        Z_wine[mask, 1],
        alpha=0.75,
        label=name
    )

plt.xlabel("PC1 score")
plt.ylabel("PC2 score")
plt.title("Wine data projected onto the first two PCs")
plt.legend(title="class")

plt.tight_layout()
plt.show()

### Interpreting loadings

The loadings are the entries of the vectors $v_i$.

For PC1,

$$
z_1=Xv_1=X_1v_{11}+X_2v_{12}+\cdots+X_Dv_{1P}.
$$

Variables with large absolute loading values contribute more to that principal component. The sign is meaningful relative to other variables, but the whole vector can be multiplied by $-1$ without changing the PCA solution. Sometimes we can "interpret" what the PCs are telling us:

In [ ]:
pc1_loadings = pd.Series(
    wine_components[:, 0],
    index=feature_names,
    name="PC1 loading"
)

pc2_loadings = pd.Series(
    wine_components[:, 1],
    index=feature_names,
    name="PC2 loading"
)

loading_table = pd.concat([pc1_loadings, pc2_loadings], axis=1)

loading_table["abs PC1 loading"] = loading_table["PC1 loading"].abs()

loading_table = loading_table.sort_values(
    "abs PC1 loading",
    ascending=False
)

display(loading_table)

In [ ]:
loading_table["PC1 loading"].sort_values().plot(kind="barh", figsize=(7, 5))
plt.xlabel("loading value")
plt.title("PC1 loadings for standardized wine variables")
plt.show()

### Note on matrix decompositions

The eigenvalue/eigenvector approach above is the PCA-level version of ideas that are often written using the EVD or SVD in a linear algebra course; for this lecture, the covariance matrix and its eigenvectors are enough.

## High-dimensional Visualization: MNIST

We can also use PCA to visualize high dimensional data like MNIST:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X = mnist.data.astype(float)
y = mnist.target.astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

rng = np.random.default_rng(657677)

n_sample = 10000
idx = rng.choice(X.shape[0], size=n_sample, replace=False)

X_sub = X[idx]
y_sub = y[idx]

# Scale pixels to [0, 1]
X_sub = X_sub / 255.0

In [ ]:
# PCA centers internally, so we do not need to manually subtract the mean.
pca = PCA(n_components=2)
Z = pca.fit_transform(X_sub)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

print("\nTotal variance explained by first 2 PCs:")
print(pca.explained_variance_ratio_.sum().round(4))

In [ ]:
plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    Z[:, 0],
    Z[:, 1],
    c=y_sub,
    s=8,
    alpha=0.75,
    cmap="tab10"
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("MNIST projected onto the first two principal components")

cbar = plt.colorbar(scatter, ticks=np.arange(10))
cbar.set_label("Digit")

plt.tight_layout()
plt.show()

# Review Questions

See: @sec-pca-questions.